[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/fluids/mhd_sphere/mhd_sphere.ipynb)

# Magnetohydrodynamics on a Sphere

This example simulates magnetohydrodynamics in a thin spherical shell of conducting fluid whose flow winds up an embedded magnetic field while magnetic tension pushes back on the flow. Because a conducting fluid drags magnetic field lines along with its motion, differential shear stretches and amplifies the field until bent field lines pull back like taut strings and deflect the flow. Where oppositely directed field lines are pressed together, electric current concentrates into thin sheets that reconnect into closed loops called magnetic islands. Differential rotation between the equator and middle latitudes winds an initially pole-to-pole magnetic field into bands around the sphere.

In four-dimensional spacetime at a single instant, points on the unit sphere are modelled as spatial directions, and joining two edges of a triangle forms its oriented area patch `(sides[0] & sides[1]) / 2`. Incompressible surface flow and source-free surface magnetic flux are each represented by a vertex scalar, the stream function and the flux function, whose exterior meet with the triangle patch `gradient(stream) ^ patches` turns a gradient plane into tangent area-weighted flow along contours. Pairing another gradient with that flow `-(gradient(function) & flows)` integrates their surface bracket over each triangle, expressing both frozen-in magnetic advection along fluid worldlines and the Lorentz torque between flux and current while conserving total energy, cross helicity, and mean-square flux in the ideal limit.

In Gibbs–Heaviside vector notation on the unit sphere with outward normal $\hat{\mathbf r}$, skew-gradient velocities $\mathbf u = \hat{\mathbf r} \times \nabla_s \phi$ and magnetic fields $\mathbf B = \hat{\mathbf r} \times \nabla_s \psi$ interact through the surface Jacobian $J(a, b) = \hat{\mathbf r} \cdot (\nabla_s a \times \nabla_s b)$, giving $\partial_t \omega + J(\phi, \omega) = J(\psi, j) + \nu \nabla_s^2 \omega$ and $\partial_t \psi + J(\phi, \psi) = \eta \nabla_s^2 \psi$ with $\omega = -\nabla_s^2 \phi$ and $j = -\nabla_s^2 \psi$.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterable, Iterator
from itertools import islice, tee

import numpy as np

from numga import Algebra, NumpyContext, stack
from numga.algebras import STA as ga
from numga.sparse import spdiag
from examples.util import shapes
from examples.util.animation import save_figure, show_animation
from examples.fluids.mhd_sphere import render

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                   # Scalar+
Antiscalar = ga.gatype.antiscalar()                           # Antiscalar-
# Spatial planes read coordinates on the sphere and represent triangle gradients.
Plane = ga.gatype.vector().derive.within(Algebra("x+y+z+"))   # Vector+
# Spatial directions represent sphere vertices, triangle edges, and tangent velocities.
Point = Plane.derive.dual                                     # Antivector-
# Joining two triangle edges forms an oriented surface patch.
Patch = ga.gatype.antibivector()                              # Antibivector-

## Simplex gradients and surface operators

Each triangle's boundary edges are spatial directions whose join forms its oriented area patch. The term ***simplex gradient*** here means the spatial plane on a triangle that rises from zero along one edge to one at the opposite vertex, computed from the opposite edge and triangle patch as `(sides | patches.inverse()) / 2`. Meeting a vertex function's gradient with the unit triangle patch turns contour slopes into tangent velocities, and pulling momentum back along `velocity` forms the Dirichlet stiffness `(Antiscalar- <- Scalar+)[V <- V]`. Pinning one reference vertex fixes the stream function's free additive constant without altering any velocity.

In [ ]:
n_levels = 5

In [ ]:
topology, coordinates = shapes.sphere(n_levels)
V, E, F = topology.n_elements
vertices = mv(Plane, coordinates).dual().field()                                 # Point-[V]
sides = topology.boundary[2, 1] * (topology.d[0] * vertices)                     # [3] Point-[F]
patches = (sides[0] & sides[1]) / 2                                              # Patch-[F]
areas = patches.antinorm()                                                       # Scalar+[F]
gradients = (sides | patches.inverse()) / 2                                      # [3] Plane+[F]
gradient = (spdiag(gradients * Scalar) * topology.incidence[2, 0]).sum(axis=0)   # (Plane+ <- Scalar+)[F <- V]
velocity = (Plane ^ (patches / areas))(gradient)                                 # (Point- <- Scalar+)[F <- V]

# Each vertex dual cell receives one third of each incident triangle's area and integral.
corner_mean = topology.incidence[2, 0].mean(axis=0)                              # Scalar+[F <- V]
dual_areas = corner_mean.adjugate() * areas                                      # Scalar+[V]
vertex_projection = spdiag(1 / dual_areas) * corner_mean.adjugate()              # Scalar+[V <- F]
momentum = areas * Point.dual()                                                  # (Plane+ <- Point-)[F]
stiffness = velocity.adjugate()(momentum(velocity))                              # (Antiscalar- <- Scalar+)[V <- V]
mass = spdiag(dual_areas * Scalar.dual())                                        # (Antiscalar- <- Scalar+)[V <- V]
reference = mv.scalar((np.arange(V) == 0)[:, None]).field()                      # Scalar+[V]
pinned_stiffness = stiffness + stiffness.diagonal() * reference                  # (Antiscalar- <- Scalar+)[V <- V]

## Surface bracket and frozen-in induction

An incompressible flow advects a vertex function at minus the pairing of that function's gradient with the area-weighted tangent flow `gradient(stream) ^ patches`, projected onto dual cells. Fluid shear advects vorticity through `bracket(stream, vorticity)` while bent magnetic field lines apply an opposing Lorentz torque `bracket(flux, current)`. Freezing magnetic flux into fluid worldlines sets each triangle's ideal flux advection to `-(gradient(flux) & flows)`, while resistivity diffuses flux proportional to the electric current density.

In [ ]:
def bracket(stream: Scalar, function: Scalar) -> Scalar:
    """Triangle-integrated pairing of `function`'s gradient with `stream`'s tangent flow projects onto dual cells as the vertex advection rate (`Scalar+[V]`)."""
    flows = gradient(stream) ^ patches                                           # Point-[F]
    return -(vertex_projection * (gradient(function) & flows))                   # Scalar+[V]


def laplacian(function: Scalar) -> Scalar:
    """Solving the lumped dual-cell mass against Dirichlet stiffness evaluates the positive-definite negated surface Laplacian (`Scalar+[V]`)."""
    return mass.solve(stiffness(function))                                       # Scalar+[V]

In [ ]:
def rate(state: Scalar, viscosity: float, resistivity: float) -> Scalar:
    """Vorticity advection, Lorentz tension, frozen-in flux advection, and diffusive damping determine the time derivative (`[2] Scalar+[V]`) of `state` (`[2] Scalar+[V]`)."""
    vorticity, flux = state                                                      # Scalar+[V], Scalar+[V]
    stream = pinned_stiffness.solve(mass(vorticity))                             # Scalar+[V]
    current = laplacian(flux)                                                    # Scalar+[V]
    vorticity_rate = bracket(stream, vorticity) - bracket(flux, current) - viscosity * laplacian(vorticity)  # Scalar+[V]
    # Vanishing pairing of the spacetime flux gradient along fluid worldlines sets ideal advection, slipped by current.
    flows = gradient(stream) ^ patches                                           # Point-[F]
    advection = -(gradient(flux) & flows)                                        # Scalar+[F]
    flux_rate = vertex_projection * advection - resistivity * current            # Scalar+[V]
    return stack((vorticity_rate, flux_rate))                                    # [2] Scalar+[V]


def evolution(initial: Scalar, viscosity: float, resistivity: float, dt: float, n_steps: int) -> Iterator[Scalar]:
    """Classical fourth-order Runge–Kutta integration yields `n_steps + 1` successive vorticity-flux states (`[2] Scalar+[V]`) starting from `initial`."""
    state = initial                                                              # [2] Scalar+[V]
    yield state
    for _ in range(n_steps):
        first_rate = rate(state, viscosity, resistivity)                         # [2] Scalar+[V]
        second_rate = rate(state + (dt / 2) * first_rate, viscosity, resistivity)   # [2] Scalar+[V]
        third_rate = rate(state + (dt / 2) * second_rate, viscosity, resistivity)   # [2] Scalar+[V]
        fourth_rate = rate(state + dt * third_rate, viscosity, resistivity)         # [2] Scalar+[V]
        state = state + (dt / 6) * (first_rate + 2 * second_rate + 2 * third_rate + fourth_rate)  # [2] Scalar+[V]
        yield state

## Differential rotation winding up the field

The initial flow is a degree-three zonal stream function in which the equator rotates opposite to the middle latitudes, while the initial magnetic field follows the pole-to-pole contours of `x & vertices` around the two equatorial poles on the x-axis. Because initial kinetic energy exceeds magnetic energy by roughly fifty to one, shear at first winds the field nearly passively. Stepping the sphere takes about half a minute because each fourth-order Runge–Kutta step solves the pinned Poisson system four times, streaming rendered frames as they finish.

In [ ]:
rotation = 1.0
field_strength = 0.25
viscosity = 1e-3
resistivity = 1e-3
dt = 0.02
n_steps = 450
stride = 4
n_frames = n_steps // stride + 1
frame_duration_ms = 40

In [ ]:
heights = mv.z & vertices                                                        # Scalar+[V]
initial_stream = rotation * (5 * heights * heights * heights - 3 * heights) / 2  # Scalar+[V]
initial_flux = field_strength * (mv.x & vertices)                                # Scalar+[V]
initial_state = stack((laplacian(initial_stream), initial_flux))                 # [2] Scalar+[V]
animated_states, recorded_states = tee(
    islice(evolution(initial_state, viscosity, resistivity, dt, n_steps), 0, None, stride)
)


def frame_channels(states: Iterable[Scalar]) -> Iterator[tuple[Scalar, Scalar, Scalar]]:
    """Each vorticity-flux state yields vertex current density (`Scalar+[V]`), vertex flux (`Scalar+[V]`), and triangle field strength (`Scalar+[F]`)."""
    for state in states:
        flux = state[1]                                                          # Scalar+[V]
        yield laplacian(flux), flux, gradient(flux).norm()


show_animation(render.animation(topology, vertices, frame_channels(animated_states)), "mhd_sphere", frame_duration_ms,
               n_frames=n_frames);

Orange and violet shading shows positive and negative radial current sheet strength on the sphere, overlaid with dark magnetic field lines spaced evenly in initial flux. Differential rotation stretches the poloidal lines into azimuthal bands until oppositely directed bands press together into thin current sheets, where resistive reconnection pinches off magnetic islands that subsequently merge.

## Energy exchange between flow and field

Stretching magnetic field lines transfers kinetic energy from the shear flow into magnetic energy until magnetic tension drives the flow back, while viscosity and resistivity steadily dissipate both. Pairing each function with its Dirichlet stiffness image `streams & stiffness(streams)` and `fluxes & stiffness(fluxes)` integrates the squared velocity and magnetic field magnitudes over the sphere.

In [ ]:
states = stack(tuple(recorded_states))                                           # [n_frames, 2] Scalar+[V]
vorticities, fluxes = states[:, 0], states[:, 1]                                 # [n_frames] Scalar+[V], [n_frames] Scalar+[V]
streams = pinned_stiffness.solve(mass(vorticities))                              # [n_frames] Scalar+[V]
kinetic = (streams & stiffness(streams)).sites.sum() / 2                         # [n_frames] Scalar+
magnetic = (fluxes & stiffness(fluxes)).sites.sum() / 2                          # [n_frames] Scalar+
times = np.arange(n_frames) * (stride * dt)
save_figure(render.energy_figure(times, kinetic, magnetic), "mhd_sphere_energies");

The blue curve tracks kinetic energy and the orange curve tracks magnetic energy as fractions of the initial total energy. Differential rotation converts kinetic energy into wound-up magnetic energy before Lorentz tension returns part of that energy to the flow, while resistive current sheets and viscous shear monotonically lower the combined total.

In [ ]:
# checks
# Ideal rates conserve total energy, cross helicity, and mean-square flux via bracket antisymmetry.
vorticity, flux = states[n_frames // 4]                                          # Scalar+[V], Scalar+[V]
vorticity_rate, flux_rate = rate(stack((vorticity, flux)), 0.0, 0.0)             # Scalar+[V], Scalar+[V]
stream = pinned_stiffness.solve(mass(vorticity))                                 # Scalar+[V]
current = laplacian(flux)                                                        # Scalar+[V]
energy_rate = ((stream & mass(vorticity_rate)) + (current & mass(flux_rate))).sites.sum()          # Scalar+
cross_helicity_rate = ((flux & mass(vorticity_rate)) + (vorticity & mass(flux_rate))).sites.sum()  # Scalar+
flux_square_rate = (flux & mass(flux_rate)).sites.sum()                                            # Scalar+
exchange_scale = (current & mass(bracket(stream, flux))).sites.sum().abs()                         # Scalar+
for invariant_rate in (energy_rate, cross_helicity_rate, flux_square_rate):
    assert (invariant_rate.abs() < 1e-11 * exchange_scale).all()
total_energy = kinetic + magnetic                                                # [n_frames] Scalar+
assert (total_energy[1:] < total_energy[:-1]).all()